### Create basic tsv of lang meta info

In [12]:
from iso639 import Lang

In [13]:
import os

fleurs_dir = "fleurs/data/"
lang_dirs = os.listdir(fleurs_dir)

In [16]:
lang_meta = {'dir': [], 'name':[], 'iso639-1': [], 'iso639-3': [], 'script': []}
for dr in lang_dirs:
    pth = os.path.join(fleurs_dir, dr)
    if os.path.isdir(pth):
        name = dr.split('_')
        lang = Lang(name[0])
        lang_meta['dir'].append(pth)
        lang_meta['name'].append(lang.name)
        lang_meta['iso639-1'].append(lang.pt1)
        lang_meta['iso639-3'].append(lang.pt3)
        if len(name) == 3:
            lang_meta['script'].append(name[1])
        else:
            lang_meta['script'].append('')
        

In [17]:
import pandas as pd

lang_meta_df = pd.DataFrame(lang_meta)
lang_meta_df

,dir,name,iso639-1,iso639-3,script
0,fleurs/data/af_za,Afrikaans,af,afr,
1,fleurs/data/am_et,Amharic,am,amh,
2,fleurs/data/ar_eg,Arabic,ar,ara,
3,fleurs/data/as_in,Assamese,as,asm,
4,fleurs/data/ast_es,Asturian,,ast,
...,...,...,...,...,...
97,fleurs/data/wo_sn,Wolof,wo,wol,
98,fleurs/data/xh_za,Xhosa,xh,xho,
99,fleurs/data/yo_ng,Yoruba,yo,yor,
100,fleurs/data/yue_hant_hk,Yue Chinese,,yue,hant


In [19]:
lang_meta_df = pd.read_csv("lang_meta.tsv", sep='\t').fillna('')

In [20]:
lang_meta_df

,dir,name,iso639-1,iso639-3,script,northeuralex,dict,g2p,phoneset,acoustic
0,fleurs/data/af_za,Afrikaans,af,afr,,,,,,
1,fleurs/data/am_et,Amharic,am,amh,,,,,,
2,fleurs/data/ar_eg,Arabic,ar,ara,,,arabic_mfa,,,
3,fleurs/data/as_in,Assamese,as,asm,,,,,,
4,fleurs/data/ast_es,Asturian,,ast,,,,,,
...,...,...,...,...,...,...,...,...,...,...
97,fleurs/data/wo_sn,Wolof,wo,wol,,,,,,
98,fleurs/data/xh_za,Xhosa,xh,xho,,,,,,
99,fleurs/data/yo_ng,Yoruba,yo,yor,,,,,,
100,fleurs/data/yue_hant_hk,Yue Chinese,,yue,hant,,,,,


In [21]:
north_eura_lex_langs = pd.read_csv("northeuralex/northeuralex-0.9-language-data.tsv",sep='\t').iso_code.tolist()

In [25]:
lang_meta_df['northeuralex'] = lang_meta_df.apply(lambda x: x['iso639-3'] if x['iso639-3'] in north_eura_lex_langs else '', axis=1)

In [27]:
#lang_meta_df.to_csv("lang_meta.tsv", sep='\t', index=False)

### Create text files from train tsv

In [195]:
from tqdm import tqdm
import pandas as pd
import os
import csv
import unicodedata

root = "fleurs/data"
out_root = "fleurs_ipa/"
HEADERS = ['id', 'audio_file', 'text', 'text_normalized', 'chars', 'speaker_id', 'gender']
LATIN_LANG_PREFIXES = (
    "af_", "ast_", "az_", "bs_", "ca_", "ceb_", "cs_", "cy_", "da_", "de_", "en_", "es_", "et_", "ff_", "fi_", "fil_", "fr_", "ga_", "gl_", "ha_", "hr_", "hu_", "id_", "ig_", "is_", "it_", "jv_", "kam_", "kea_", "lb_", "lg_", "ln_", "lt_", "luo_", "lv_", "mi_", "ms_", "mt_", "nb_", "nl_", "nso_", "ny_", "oc_", "om_", "pl_", "pt_", "ro_", "sk_", "sl_", "sn_", "so_", "sv_", "sw_", "tr_", "umb_", "uz_", "vi_", "wo_", "xh_", "yo_", "zu_"
)

ZH_PREFIXES = ("cmn_","yue_")

def clean_text(text, lang_dir):
    # remove punctuation
    text = "".join(ch for ch in text if not unicodedata.category(ch).startswith("P"))
    text = ''.join(ch for ch in text if unicodedata.category(ch) != 'Cf')

    # collapse spaces BETWEEN CJK characters
    is_zh = lang_dir.startswith(ZH_PREFIXES)
    if is_zh:
        text = re.sub(
            r'(?<=[\u4e00-\u9fff])\s+(?=[\u4e00-\u9fff])',
            '',
            text
        )
        text = re.sub(r'[—–―─]+', ' ', text)
    is_latin_lang = lang_dir.startswith(LATIN_LANG_PREFIXES)
    # ONLY apply this for non-Latin languages
    if not is_latin_lang:
        text = re.sub(
            r'(?<=[A-Za-z\u00C0-\u024F])(?=[^A-Za-z\u00C0-\u024F\s])|(?<=[^A-Za-z\u00C0-\u024F\s])(?=[A-Za-z\u00C0-\u024F])',
            ' ',
            text
        )

    # separate digits and non-digits (safe for all)
    text = re.sub(r'(?<=\d)(?=\D)|(?<=\D)(?=\d)', ' ', text)

    # normalize whitespace
    text = " ".join(text.split())

    return text

    
for lang_dir in tqdm(os.listdir(root)):
    lang_path = os.path.join(root, lang_dir)
    if not os.path.isdir(lang_path):
        continue

    out_path = os.path.join(out_root, lang_dir)
    os.makedirs(out_path, exist_ok=True)
    tsv_path = os.path.join(lang_path, "train.tsv")
    try:
        lang_df = pd.read_csv(tsv_path, names=HEADERS, sep='\t', quoting=csv.QUOTE_NONE, encoding='utf-8')
    except:
        lang_df = pd.read_csv(tsv_path, names=HEADERS+[""], sep='\t', quoting=csv.QUOTE_NONE, encoding='utf-8')
    lang_text = lang_df['text_normalized'].tolist()
    
    lang_text = [clean_text(t, lang_dir) for t in lang_text]
    with open(os.path.join(out_path, 'train_sentences.txt'), 'w') as fp:
        fp.write('\n'.join(lang_text))

100%|█████████████████████████████████████████| 103/103 [00:18<00:00,  5.65it/s]


### Create Epitran transcriptions

In [196]:
import epitran
import pandas as pd
import os
from tqdm import tqdm
import logging

logger = logging.Logger('catch_all')

tqdm.pandas()

In [197]:
meta_pd = pd.read_csv('lang_meta.tsv', sep='\t').fillna('')

def transcribe_epi(meta_row):
    lang = os.path.basename(meta_row['dir'])
    lang_pth = os.path.join('fleurs_ipa', lang)
    epi_code = meta_row['epitran']

    if epi_code:
        try:
            if 'cmn' in epi_code:
                epi = epitran.Epitran(epi_code, cedict_file='epitran_dicts/cedict_1_0_ts_utf-8_mdbg/cedict_ts.u8', tones=True)
            elif 'yue' in epi_code:
                epi = epitran.Epitran(epi_code, cedict_file='epitran_dicts/cccanto-170202/cccanto-webdist.txt', tones=True)
            else:
                epi = epitran.Epitran(epi_code, tones=True)
        except Exception as e:
            print(epi_code)
            logger.exception(e)
            return

        with open(os.path.join(lang_pth, 'train_sentences.txt'), 'r', encoding='utf-8') as fp:
            lang_text = fp.read().split('\n')

        lang_text_transcribed = []
        lang_text_reversed = []  # NEW: for MFA input

        for line in lang_text:
            # Languages without reliable spacing
            if any(x in epi_code for x in ['cmn', 'yue']):
                # reverse merging for MFA input (CJK + Thai + Lao)
                reversed_line = re.sub(
                    r'([\u4e00-\u9fff])',
                    r' \1 ',
                    line
                )
                reversed_line = " ".join(reversed_line.split())
                lang_text_reversed.append(reversed_line)
            
                pieces = line.split()
            
                full = []
                reversed_pieces = []
            
                for p in pieces:
                    # 🔥 FIX: skip numbers
                    if p.isdigit():
                        full.append(p)
                        rev = p
                    else:
                        ipa = epi.transliterate(p)
                        full.append(ipa)
                
                        rev = re.sub(
                            r'([\u4e00-\u9fff])',
                            r' \1 ',
                            p
                        )
                        rev = " ".join(rev.split())
                
                    reversed_pieces.append(rev)
            
                syllables = []
                src_tokens = []
            
                for ipa, rev in zip(full, reversed_pieces):
                    tokens = rev.split(' ')
                    src_tokens.extend(tokens)
            
                    if not ipa:
                        continue
            
                    # detect tone: IPA tone marks OR Cantonese tone digits (0-6) attached to non-digits
                    # 🔥 Step 1: split around any remaining CJK characters
                    parts = re.split(r'([\u4e00-\u9fff])', ipa)
                    
                    for part in parts:
                        if not part:
                            continue
                    
                        # if it's a raw CJK character → keep as-is
                        if re.match(r'^[\u4e00-\u9fff]$', part):
                            syllables.append(part)
                            continue
                    
                        # 🔥 Step 2: normal tone-based splitting
                        if re.search(r'[˥˧˨˩˦]|[^\d\s]+[0-6]', part):
                            syllables.extend(
                                re.findall(r'[^\d\s˥˧˨˩˦]+[˥˧˨˩˦]+|[^\d\s˥˧˨˩˦]+[0-6]', part)
                            )
                        else:
                            syllables.append(part)
                BAD_BIGRAMS = {
                    ('pɔw3', 'syu4'): 'pɔw3',
                    ('niw6', 'doi6'): 'niw6',
                }
                
                cleaned_syllables = []
                i = 0
                while i < len(syllables):
                    if i < len(syllables) - 1:
                        pair = (syllables[i], syllables[i + 1])
                        if pair in BAD_BIGRAMS:
                            cleaned_syllables.append(BAD_BIGRAMS[pair])
                            i += 2
                            continue
                
                    cleaned_syllables.append(syllables[i])
                    i += 1
                
                syllables = [s for s in cleaned_syllables if not s.startswith('*')]
                if len(src_tokens) == len(syllables):
                    lang_text_transcribed.append(' '.join(syllables))
                else:
                    # piecewise fallback (critical fix)
                    fallback = []
                    print("Mismatch:")
                    print("SRC:", src_tokens)
                    print("IPA:", syllables)
                    fallback = [ipa for ipa in full if ipa]
                    lang_text_transcribed.append(' '.join(fallback))

            else:
                lang_text_transcribed.append(epi.transliterate(line))
                lang_text_reversed.append(line)

        # write transcription
        with open(os.path.join(lang_pth, 'train_sentences_epitran.txt'), 'w', encoding='utf-8') as fp:
            fp.write('\n'.join(lang_text_transcribed))

        # NEW: write reversed input for MFA
        with open(os.path.join(lang_pth, 'train_sentences_input.txt'), 'w', encoding='utf-8') as fp:
            fp.write('\n'.join(lang_text_reversed))
    

In [198]:
meta_pd.progress_apply(transcribe_epi, axis=1)

 17%|███████                                   | 17/102 [00:15<01:58,  1.39s/it]Undefined symbol: ::consonant::
Traceback (most recent call last):
  File "/tmp/ipykernel_1440597/909046445.py", line 15, in transcribe_epi
    epi = epitran.Epitran(epi_code, tones=True)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/mahesh-akavarapu/anaconda3/lib/python3.12/site-packages/epitran/_epitran.py", line 51, in __init__
    self.epi = SimpleEpitran(code, **kwargs)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/mahesh-akavarapu/anaconda3/lib/python3.12/site-packages/epitran/simple.py", line 66, in __init__
    self.preprocessor = PrePostProcessor(code, 'pre', False)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/mahesh-akavarapu/anaconda3/lib/python3.12/site-packages/epitran/ppprocessor.py", line 27, in __init__
    self.rules = self._read_rules(code, fix, rev)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/mahesh-akavarapu/a

dan-Latn


100%|█████████████████████████████████████████| 102/102 [04:11<00:00,  2.46s/it]


0      None
1      None
2      None
3      None
4      None
       ... 
97     None
98     None
99     None
100    None
101    None
Length: 102, dtype: object

### Create pronunciation dictionaries

In [199]:
from lingpy import ipa2tokens
import re

LATIN_RE = re.compile(r'^[a-z]+$')


def build_lexicon_entry(src_sent: str, tar_sent: str):
    """
    Build a partial pronunciation dictionary from a source sentence
    and its G2P-transcribed target sentence.

    Returns:
        dict: {word: set(pronunciations)}
    """

    src_tokens = re.sub(r"\s", " ", src_sent).split(' ')
    tar_tokens = re.sub(r"\s", " ", tar_sent).split(' ')

    lexicon = {}

    if len(src_tokens) != len(tar_tokens):
        raise ValueError("Source and target token lengths do not match")

    for src_tok, tar_tok in zip(src_tokens, tar_tokens):

        # Case 0: empty pronunciation (e.g., silent word)
        if tar_tok == "":
            lexicon.setdefault(src_tok, set()).add("spn")
            continue

        # Case 1: obvious failure marker (e.g., @@@@)
        if '@' in tar_tok:
            lexicon.setdefault(src_tok, set()).add("spn")
            continue

        # Case 2: entirely digits → treat as unknown
        if src_tok.isdigit():
            lexicon.setdefault(src_tok, set()).add("spn")
            continue

        # Case 3: pure latin lowercase ASCII
        if LATIN_RE.match(src_tok):
            tokens = ipa2tokens(tar_tok, merge_vowels=False)
            lexicon.setdefault(src_tok, set()).add(' '.join(tokens))
            continue
        
        # Case 4: transliteration failed (non-latin unchanged or leaked)
        if src_tok == tar_tok:
            lexicon.setdefault(src_tok, set()).add("spn")
            continue

        # Case 5: non-latin scripts
        tokens = ipa2tokens(tar_tok, merge_vowels=False)
        lexicon.setdefault(src_tok, set()).add(' '.join(tokens))

    return lexicon

In [200]:
src = "35mm ఆకృతి వాస్తవానికి కొంత గందరగోళంగా 36mm వెడల్పు 24mm ఎత్తు"
tar1 = "@@@@ aːkrut̪i waːst̪awaːniki komt̪a ɡamd̪araɡoːɭamɡaː @@@@ weɖalpu @@@@ et̪ːu"
tar2 = "35mm akrut̪ɪ ʋast̪əʋanɪkɪ kont̪ə ɡənd̪ərəɡoːɭəŋɡa 36mm ʋeɖəlpʊ 24mm et̪t̪u"

In [201]:
src = "la fosse est soit chauffée avec des pierres chaudes provenant dun feu soit la chaleur géothermique réchauffe naturellement certaines zones du sol" 
tar = "la fɔsə  swat ʃofe avək de piɛrrə ʃodə prɔvənɑ̃t dœ̃ fœ swat la ʃalœr ʒeɔtɛrmikə reʃofə natyrələmɑ̃ sɛrtɛnə zɔn dy sɔl"

In [202]:
build_lexicon_entry(src, tar)

{'la': {'l a'},
 'fosse': {'f ɔ s ə'},
 'est': {'spn'},
 'soit': {'s w a t'},
 'chauffée': {'ʃ o f e'},
 'avec': {'a v ə k'},
 'des': {'d e'},
 'pierres': {'p i ɛ rr ə'},
 'chaudes': {'ʃ o d ə'},
 'provenant': {'p r ɔ v ə n ɑ̃ t'},
 'dun': {'d œ̃'},
 'feu': {'f œ'},
 'chaleur': {'ʃ a l œ r'},
 'géothermique': {'ʒ e ɔ t ɛ r m i k ə'},
 'réchauffe': {'r e ʃ o f ə'},
 'naturellement': {'n a t y r ə l ə m ɑ̃'},
 'certaines': {'s ɛ r t ɛ n ə'},
 'zones': {'z ɔ n'},
 'du': {'d y'},
 'sol': {'s ɔ l'}}

In [203]:
import os

def build_mfa_dictionary(src_text_path: str, tar_text_path: str, output_name="lexicon.txt"):
    """
    Build an MFA-compatible pronunciation dictionary from parallel text files.
    """

    out_dir = os.path.dirname(src_text_path) or os.path.dirname(tar_text_path)
    output_path = os.path.join(out_dir, output_name)

    lexicon = {}

    with open(src_text_path, 'r', encoding='utf-8') as f_src, \
         open(tar_text_path, 'r', encoding='utf-8') as f_tar:

        for line_num, (src_line, tar_line) in enumerate(zip(f_src, f_tar), 1):
            entry = build_lexicon_entry(src_line, tar_line)

            # merge sets (IMPORTANT CHANGE)
            for word, prons in entry.items():
                if word not in lexicon:
                    lexicon[word] = set()
                lexicon[word].update(prons)

    # Ensure <unk> exists
    if "<unk>" not in lexicon:
        lexicon["<unk>"] = {"spn"}

    # Write dictionary (tab-separated, multiple pronunciations)
    with open(output_path, 'w', encoding='utf-8') as f_out:
        for word in sorted(lexicon.keys()):
            for pron in lexicon[word]:
                f_out.write(f"{word}\t{pron}\n")

    return output_path

In [204]:
build_mfa_dictionary('fleurs_ipa/te_in/train_sentences.txt', 'fleurs_ipa/te_in/train_sentences_epitran.txt', output_name="lexicon.txt")

'fleurs_ipa/te_in/lexicon.txt'

In [205]:
import pandas as pd
from tqdm import tqdm

tqdm.pandas()
meta_pd = pd.read_csv('lang_meta.tsv', sep='\t').fillna('')

def build_pron_dict(meta_row):
    lang = os.path.basename(meta_row['dir'])
    lang_pth = os.path.join('fleurs_ipa', lang)
    epi_code = meta_row['epitran']
    xpf = meta_row['xpf']
    src_pre = "train_sentences"
    src_pth = os.path.join(lang_pth, f"{src_pre}_input.txt")
    tar_pth = ""
    if epi_code:
        tar_pth = os.path.join(lang_pth, f"{src_pre}_epitran.txt")
    elif xpf:
        if not (tar_pth and os.path.isfile(tar_pth)):
            tar_pth = os.path.join(lang_pth, f"{src_pre}_xpf.txt")

    if os.path.isfile(tar_pth):
        build_mfa_dictionary(src_pth, tar_pth)

In [206]:
meta_pd.progress_apply(build_pron_dict, axis=1)

100%|█████████████████████████████████████████| 102/102 [00:18<00:00,  5.38it/s]


0      None
1      None
2      None
3      None
4      None
       ... 
97     None
98     None
99     None
100    None
101    None
Length: 102, dtype: object